## PDF Document Loaders
- Load various kind of documents from the web and local files.
- Apply LLM to the documents for summarization and question answering.

### Project 1: Question Answering from PDF Document
- We will load the document from the local file and apply LLM to answer the questions.
- Lets use research paper published on the missuse of the health supplements for workout. 

rag-dataset: git@github.com:laxmimerit/rag-dataset.git

```bash
git clone git@github.com:laxmimerit/rag-dataset.git
```

In [21]:
!git clone https://github.com/laxmimerit/rag-dataset.git
# !pip install pymupdf tiktoken 

fatal: destination path 'rag-dataset' already exists and is not an empty directory.


In [22]:
from dotenv import load_dotenv

load_dotenv('./../.env')

True

In [23]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("rag-dataset/health supplements/1. dietary supplements - for whom.pdf")

docs = loader.load()

In [24]:
len(docs)

17

In [25]:

print(docs[0].page_content)

International  Journal  of
Environmental Research
and Public Health
Review
Dietary Supplements—For Whom? The Current State of
Knowledge about the Health Effects of Selected
Supplement Use
Regina Ewa Wierzejska


Citation: Wierzejska, R.E. Dietary
Supplements—For Whom? The
Current State of Knowledge about the
Health Effects of Selected Supplement
Use. Int. J. Environ. Res. Public Health
2021, 18, 8897. https://doi.org/
10.3390/ijerph18178897
Academic Editor: Paul B. Tchounwou
Received: 15 July 2021
Accepted: 21 August 2021
Published: 24 August 2021
Publisher’s Note: MDPI stays neutral
with regard to jurisdictional claims in
published maps and institutional afﬁl-
iations.
Copyright: © 2021 by the author.
Licensee MDPI, Basel, Switzerland.
This article is an open access article
distributed
under
the
terms
and
conditions of the Creative Commons
Attribution (CC BY) license (https://
creativecommons.org/licenses/by/
4.0/).
Department of Nutrition and Nutritional Value of Food, 

In [26]:
docs[0].metadata

{'producer': 'iLovePDF',
 'creator': '',
 'creationdate': '',
 'source': 'rag-dataset/health supplements/1. dietary supplements - for whom.pdf',
 'file_path': 'rag-dataset/health supplements/1. dietary supplements - for whom.pdf',
 'total_pages': 17,
 'format': 'PDF 1.7',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2024-10-21T11:37:54+00:00',
 'trapped': '',
 'modDate': 'D:20241021113754Z',
 'creationDate': '',
 'page': 0}

In [27]:
### Read the list of PDFs in the dir
import os

pdfs = []
for root, dirs, files in os.walk("rag-dataset"):
    # print(root, dirs, files)
    for file in files:
        if file.endswith(".pdf"):
            pdfs.append(os.path.join(root, file))

In [28]:
docs = []
for pdf in pdfs:
    loader = PyMuPDFLoader(pdf)
    temp = loader.load()
    docs.extend(temp)

    # print(temp)
    # break

In [29]:
len(docs)

1760

In [30]:
def format_docs(docs):
    return "\n\n".join([x.page_content for x in docs])


context = format_docs(docs)

In [31]:
docs[0]

Document(metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2024-02-02T06:04:36-05:00', 'source': 'rag-dataset\\finance\\pdfs\\amazon\\amazon 10-k 2023.pdf', 'file_path': 'rag-dataset\\finance\\pdfs\\amazon\\amazon 10-k 2023.pdf', 'total_pages': 94, 'format': 'PDF 1.4', 'title': '0001018724-24-000008', 'author': 'EDGAR® Online LLC, a subsidiary of OTC Markets Group', 'subject': 'Form 10-K filed on 2024-02-02 for the period ending 2023-12-31', 'keywords': '0001018724-24-000008; ; 10-K', 'moddate': '2024-02-02T06:04:51-05:00', 'trapped': '', 'encryption': 'Standard V2 R3 128-bit RC4', 'modDate': "D:20240202060451-05'00'", 'creationDate': "D:20240202060436-05'00'", 'page': 0}, page_content='Table of Contents\nUNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\n\xa0____________________________________\nFORM 10-K\n____________________________________\xa0\n(Mark One)\n☒\nANNUAL REPORT PURSUANT TO SECT

In [32]:
# print(context)

In [33]:
#To estimate the number of tokens in the context, we can use tiktoken library
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4o-mini")


In [34]:
encoding.encode("congratulations"), encoding.encode("rqsqeft")

([542, 111291, 14571], [81, 31847, 80, 5276])

We can see three tokens for "congratulations"

In [35]:
len(encoding.encode(docs[0].page_content))

1115

In [36]:
len(encoding.encode(context))

1233700

In [37]:
1115*64

71360

In [38]:
### Question Answering using LLM
from langchain_ollama import ChatOllama

from langchain_core.prompts import (SystemMessagePromptTemplate, HumanMessagePromptTemplate,
                                    ChatPromptTemplate)



from langchain_core.output_parsers import StrOutputParser

base_url = "http://localhost:11434"
model = 'qwen3'

llm = ChatOllama(base_url=base_url, model=model)


In [50]:
system = SystemMessagePromptTemplate.from_template("""You are helpful AI assistant who answer user question based on the provided context. 
                                                    Do not answer in more than {words} words""")

prompt = """Answer user question based on the provided context ONLY! If you do not know the answer, just say "I don't know".
            ### Context:
            {context}

            ### Question:
            {question}

            ### Answer:"""

prompt = HumanMessagePromptTemplate.from_template(prompt)

messages = [system, prompt]
template = ChatPromptTemplate(messages)

# template
# template.invoke({'context': context, 'question': "How to gain muscle mass?", 'words': 50})

qna_chain = template | llm | StrOutputParser()

In [51]:
qna_chain

ChatPromptTemplate(input_variables=['context', 'question', 'words'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['words'], input_types={}, partial_variables={}, template='You are helpful AI assistant who answer user question based on the provided context. \n                                                    Do not answer in more than {words} words'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer user question based on the provided context ONLY! If you do not know the answer, just say "I don\'t know".\n            ### Context:\n            {context}\n\n            ### Question:\n            {question}\n\n            ### Answer:'), additional_kwargs={})])
| ChatOllama(model='qwen3', base_url='http://localhost:11434')
| StrOutputParser()

In [ ]:
response = qna_chain.invoke({'context': context, 'question': "How to gain muscle mass?", 'words': 50})
print(response)

In [ ]:
response = qna_chain.invoke({'context': context, 'question': "How to reduce the weight?", 'words': 50})
print(response)

Reducing weight effectively and sustainably requires a combination of dietary adjustments, physical activity, lifestyle changes, and behavioral strategies. Here’s a structured approach:

---

### **1. Create a Calorie Deficit**
- **Eat fewer calories than you burn**: Aim for a moderate deficit (e.g., 500–750 kcal/day) to lose 0.5–1 kg/week.  
- **Focus on nutrient-dense foods**: Prioritize whole foods like vegetables, fruits, lean proteins, whole grains, and healthy fats.  
- **Limit processed foods**: Reduce sugary snacks, refined carbs (white bread, pastries), and high-fat processed items.  
- **Track intake**: Use apps like MyFitnessPal or a food journal to monitor calories and identify patterns.

---

### **2. Adopt a Balanced Diet**
- **Increase protein**: Protein boosts satiety and metabolism. Include eggs, fish, poultry, legumes, and plant-based sources.  
- **Choose complex carbs**: Opt for whole grains (brown rice, oats), starchy vegetables (sweet potatoes), and legumes.  
- *

In [43]:
response = qna_chain.invoke({'context': context, 'question': "How to do weight loss?", 'words': 50})
print(response)

Weight loss requires a combination of healthy lifestyle changes, including diet, exercise, and behavioral adjustments. While some dietary supplements may support weight loss efforts, they should not replace foundational strategies. Here’s a structured approach:

---

### **1. Focus on a Balanced Diet**
- **Calorie Deficit**: Consume fewer calories than you burn. Prioritize whole, unprocessed foods like vegetables, fruits, lean proteins, whole grains, and healthy fats.
- **Protein Intake**: Include adequate protein (e.g., eggs, fish, legumes) to preserve muscle mass and boost satiety.
- **Limit Processed Foods**: Reduce added sugars, refined carbs, and trans fats. Avoid sugary drinks and high-calorie snacks.
- **Portion Control**: Use smaller plates, avoid overeating, and eat slowly to enhance fullness.

---

### **2. Incorporate Regular Physical Activity**
- **Aerobic Exercise**: Engage in activities like walking, cycling, or swimming for at least 150 minutes weekly to burn calories.
-

In [ ]:
response = qna_chain.invoke({'context': context, 'question': "How many planets are there outside of our solar system?", 'words': 50})
print(response)

As of the latest data in 2023, there are **over 5,000 confirmed exoplanets** (planets outside our solar system) discovered by astronomers. This number is continually increasing as new exoplanets are detected using methods like the transit method, radial velocity, and direct imaging. 

However, the exact count can vary slightly depending on the source, as new discoveries are made regularly. Additionally, many more exoplanets are suspected but not yet confirmed, especially with ongoing missions like NASA's **TESS (Transiting Exoplanet Survey Satellite)** and the **James Webb Space Telescope (JWST)**, which are expanding our ability to detect and characterize distant worlds. 

In summary:  
**Confirmed exoplanets: >5,000 (as of 2023)**  
**Estimated total (including unconfirmed candidates): Likely in the tens of thousands or more**.


### Project 2: PDF Document Summarization

In [45]:
system = SystemMessagePromptTemplate.from_template("""You are helpful AI assistant who works as document summarizer. 
                                                   You must not hallucinate or provide any false information.""")

prompt = """Summarize the given context in {words}.
            ### Context:
            {context}

            ### Summary:"""

prompt = HumanMessagePromptTemplate.from_template(prompt)

messages = [system, prompt]
template = ChatPromptTemplate(messages)

summary_chain = template | llm | StrOutputParser()

In [46]:
summary_chain

ChatPromptTemplate(input_variables=['context', 'words'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are helpful AI assistant who works as document summarizer. \n                                                   You must not hallucinate or provide any false information.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'words'], input_types={}, partial_variables={}, template='Summarize the given context in {words}.\n            ### Context:\n            {context}\n\n            ### Summary:'), additional_kwargs={})])
| ChatOllama(model='qwen3', base_url='http://localhost:11434')
| StrOutputParser()

In [47]:
response = summary_chain.invoke({'context': context, 'words': 50})
print(response)

**Summary of Dietary Supplements and Nutraceuticals: Benefits, Risks, and Regulation**

**Overview:**  
Dietary supplements and nutraceuticals are widely consumed globally, with over 70% of Americans using them daily. The industry is a significant economic sector, valued at over $28 billion in the U.S. However, unlike drugs, supplements are not required to be FDA-approved or registered before sale, leading to potential safety and efficacy concerns.

**Regulatory Context:**  
- **FDA Regulation:** Under the Dietary Supplement Health and Education Act (DSHEA), supplements are classified as food products, not drugs. Manufacturers must ensure safety but are not required to prove efficacy. The FDA monitors adverse effects post-market, with limited pre-market oversight.  
- **International Variations:** Regulations vary by country. For example, the EU and Japan have stricter guidelines, while the U.S. allows broader use, leading to disparities in safety standards.

**Health Benefits and Evid

In [48]:
response = summary_chain.invoke({'context': context, 'words': 500})
print(response)

**Summary: Dietary Supplements and Nutraceuticals: Benefits, Risks, and Regulation**

**Overview:**  
Dietary supplements and nutraceuticals are widely consumed, with over 70% of Americans using them daily. However, they are not strictly regulated like pharmaceuticals under the **Dietary Supplement Health and Education Act (DSHEA)**, which allows marketing without pre-approval, raising safety concerns.

**Key Points:**  
1. **Regulatory Framework:**  
   - DSHEA classifies supplements as foods, not drugs, requiring minimal safety testing. Manufacturers cannot claim therapeutic benefits but can state general wellness claims.  
   - Adverse effects are monitored post-market, with limited pre-market safety evaluations.  

2. **Common Supplements and Risks:**  
   - **Vitamins/Minerals:** Excess intake (e.g., vitamin A, E) can cause toxicity (e.g., liver damage, bleeding).  
   - **Fish Oils/Omega-3s:** Generally safe but may interact with anticoagulants.  
   - **Protein Powders:** Soy-ba

### Project 3: Report Generation from PDF Document

Streamlit Tutorial: https://www.youtube.com/watch?v=hff2tHUzxJM&list=PLc2rvfiptPSSpZ99EnJbH5LjTJ_nOoSWW

In [49]:
response = qna_chain.invoke({'context': context, 
                             'question': "Provide a detailed report from the provided context. Write answer in Markdown.", 
                             'words': 2000})
print(response)

```markdown
# Adverse Effects of Nutraceuticals and Dietary Supplements

## Overview
Over 70% of Americans use dietary supplements daily, with the industry valued at over $28 billion. Unlike drugs, dietary supplements are not required to be registered or approved by the FDA prior to sale. Under the Dietary Supplement Health and Education Act (DSHEA) of 1994, the FDA can only monitor adverse reports post-marketing. While supplements are widely used, their health benefits are questionable, and they can pose significant risks, including toxicity and drug interactions.

## Key Findings

### 1. **Vitamin and Mineral Supplements**
- **Common Use**: Multivitamin/mineral supplements are the most widely used in the U.S., with 33% of adults using them. 
- **Toxicity Risks**:
  - **Vitamin A**: Excess intake can lead to bone health issues, increased fracture risk, and congenital abnormalities in pregnant women.
  - **Vitamin E**: High doses (800–1200 mg/day) may cause bleeding, diarrhea, and incr